# 阶段 3：阅读源码理解 GRPO 实现细节

> **目标**：逐段阅读 `transparent_grpo.py`（~400 行单文件），看清楚 GRPO 每一步在代码中长什么样。

## 你将学到

- 概率比 π_θ/π_θold 在代码中怎么算（取 log 相减再 exp）
- KL 散度在代码中怎么实现（K1 估计器：log_prob 相减）
- 裁剪函数在 PyTorch 中就是 `torch.clamp`
- 优势计算就是 `(reward - mean) / std`，和阶段 1 一模一样

## 前置条件

- 完成阶段 1（理解 GRPO 6 步流程）
- 完成阶段 2（用 TRL 跑过 GRPO 训练）

## 源码来源

[transparent-grpo](https://github.com/siyuan-harry/transparent-grpo)：一个 ~400 行的单文件 GRPO 实现，没有 TRL 的层层抽象，从上到下线性阅读。

## 阅读路线

1. **Cell 1**：配置类 Config —— 看参数定义
2. **Cell 2**：环境与奖励 ToyEnv —— 看奖励函数设计
3. **Cell 3**：初始化 —— 看模型加载和 Accelerator
4. **Cell 4**：收集阶段 —— 看生成、奖励、KL、优势计算
5. **Cell 5**：训练阶段 —— 看 PPO 裁剪和梯度更新
6. **Cell 6**：完整流程对比 —— 和阶段 1/2 的对应关系
7. **Cell 7**：理解检查点

## Cell 1：配置类 Config

和阶段 2 的 `GRPOConfig` 对应——这里更简单，直接用一个 Python 类定义所有参数。

### 和阶段 1/2 的参数对照

| 阶段 1 | 阶段 2 (GRPOConfig) | 阶段 3 (Config) | 含义 |
|--------|---------------------|-----------------|------|
| `G=4` | `num_generations=6` | `group_size=4` | 组大小 |
| `lr=0.05` | `learning_rate=5e-6` | `learning_rate=2e-6` | 学习率 |
| `epsilon=0.2` | `epsilon=0.2` | `clip_epsilon=0.2` | PPO 裁剪范围 |
| `beta=0.005` | `beta=0.1` | `beta=0.02` | KL 惩罚权重 |
| `num_steps=300` | `max_steps=300` | `num_steps=20` | 训练步数 |

### 新概念：`inner_update_epochs`

阶段 1 和 2 中，每批数据只更新一次参数。但 PPO/GRPO 通常对同一批数据做多次更新（inner epochs），类似「复习多遍」。

- `inner_update_epochs=3`：同一批生成的数据，反复用 3 次来更新参数
- 这也是为什么需要 PPO 裁剪——多次更新后，新策略会偏离旧策略，裁剪防止偏离太大

In [1]:
# ============================================================
# 读取 transparent_grpo.py 的配置部分
# ============================================================

source_path = 'transparent-grpo/transparent_grpo.py'

with open(source_path) as f:
    full_source = f.read()

# 显示配置类（第 20-40 行）
lines = full_source.split('\n')
print('=== Config 类（第 20-40 行）===')
for i, line in enumerate(lines[19:40], start=20):
    print(f'{i:3d} | {line}')

=== Config 类（第 20-40 行）===
 20 | class Config:
 21 |     # Model Configuration
 22 |     model_name = "Qwen/Qwen2.5-3B-Instruct"  # Change if needed
 23 |     max_new_tokens = 512                     # Length of reasoning chain
 24 |     group_size = 4                           # G: Number of samples per prompt (GRPO core)
 25 |     
 26 |     # Training Configuration
 27 |     learning_rate = 2e-6
 28 |     num_steps = 20                          # Toy example steps
 29 |     beta = 0.02                             # KL penalty (0.01 ~ 0.05 by convention) - for long CoT tasks, decrease beta to allow more exploration eg 0.005
 30 |     per_device_batch_size = 1
 31 | 
 32 |     # inner training loop configuration (for GRPO updates)
 33 |     clip_epsilon = 0.2
 34 |     inner_update_epochs = 3
 35 |     
 36 |     # System Prompt for Math Reasoning
 37 |     system_prompt = (
 38 |         "You are a helpful assistant capable of solving math problems step-by-step. "
 39 |         "Plea

## Cell 2：环境与奖励 ToyEnv

和阶段 1 的 `compute_reward()` 和阶段 2 的 `correctness_reward()` 对应。

### 这个奖励函数的特殊之处

阶段 1/2 的奖励是二元的（对=1，错=0）。但这里的奖励是**部分得分**（partial credit）：

| 步骤 | 奖励 | 说明 |
|------|------|------|
| 识别连续点 x=2, x=-2 | +0.2 | 知道在哪里检查连续性 |
| x=2 处方程正确 | +0.25 | 2a = -3 或 2a = -6 |
| 解出 a = -3 | +0.15 | 正确解出 a |
| x=-2 处方程正确 | +0.25 | -7 = -4 - b |
| 解出 b = 3 | +0.15 | 正确解出 b |
| 最终答案 a+b = 0 | +0.2 | 完整正确 |

**为什么用部分得分？** 因为如果只用 0/1 奖励，模型可能完全答错（0 分），组内全是 0 分，`frac_reward_zero_std=1.0`，学不到东西。部分得分让「接近正确」的答案也能得到一些奖励，增加组内多样性。

这和我们阶段 2 调参的经验完全一致——**让组内有区分度是 GRPO 成功的关键**。

In [2]:
# 显示 ToyEnv 类（第 45-99 行）
print('=== ToyEnv 类（第 45-99 行）===')
for i, line in enumerate(lines[44:99], start=45):
    print(f'{i:3d} | {line}')

=== ToyEnv 类（第 45-99 行）===
 45 | class ToyEnv:
 46 |     """
 47 |     A harder environment using a specific Algebra problem with partial credit rewards.
 48 |     """
 49 |     def __init__(self, tokenizer):
 50 |         self.tokenizer = tokenizer
 51 |         self.problem_text = (
 52 |             r"Let \[f(x) = \left\{\begin{array}{cl} ax+3, &\text{ if }x>2, \\ x-5 &\text{ if } -2 \le x \le 2, \\ 2x-b &\text{ if } x <-2. \end{array}\right.\]"
 53 |             r"Find $a+b$ if the piecewise function is continuous (which means that its graph can be drawn without lifting your pencil from the paper)."
 54 |         )
 55 |         self.ground_truth = "0"
 56 | 
 57 |     def sample_prompts(self, batch_size):
 58 |         # Return the same problem batch_size times to let the model learn on this specific instance
 59 |         prompts = [self.problem_text] * batch_size
 60 |         ground_truths = [self.ground_truth] * batch_size
 61 |         return prompts, ground_truths
 62 | 
 63

## Cell 3：初始化——模型加载与 Accelerator

### 两个模型

和阶段 2 一样，GRPO 需要**两个模型**：

1. **Policy Model**（可训练）：当前策略，会被梯度更新
2. **Reference Model**（冻结）：参考策略，用来计算 KL 散度，永远不更新

```python
model = AutoModelForCausalLM.from_pretrained(...)       # 可训练
ref_model = AutoModelForCausalLM.from_pretrained(...)    # 冻结
ref_model.eval()
ref_model.requires_grad_(False)  # 不计算梯度
```

### Accelerator

阶段 2 用 TRL 封装了分布式训练，这里用 HuggingFace 的 `accelerate` 库直接控制：

- `accelerator.prepare(model, optimizer, dataloader)`：自动处理分布式部署
- `accelerator.backward(loss)`：替代 `loss.backward()`，支持分布式梯度同步

### gradient_checkpointing

```python
model.gradient_checkpointing_enable()
```

这是一种用计算时间换显存的技术——不保存中间激活值，反向传播时重新计算。省显存但慢一些。阶段 2 的 TRL 内部也用了这个。

In [3]:
# 显示初始化部分（第 122-185 行）
print('=== 初始化部分（第 122-185 行）===')
for i, line in enumerate(lines[121:185], start=122):
    print(f'{i:3d} | {line}')

=== 初始化部分（第 122-185 行）===
122 | def main():
123 |     # =================================================================================
124 |     # [1] Initialization and loadings (Preparations)
125 |     # =================================================================================
126 | 
127 |     # 1.1 Initialize Accelerator for Distributed Training (Data Parallel)
128 |     accelerator = Accelerator(gradient_accumulation_steps=1, mixed_precision="bf16")
129 |     device = accelerator.device # Let Accelerate handle device placement; avoid manual set_device to prevent invalid ordinals
130 |     
131 |     # Printing Accelerator Configuration
132 |     num_gpus = accelerator.num_processes # Accelerator 会自动识别实际启动了多少个进程 (ie 使用了多少张卡)
133 |     total_batch_size_per_step = Config.per_device_batch_size * num_gpus
134 |     total_samples_per_step = total_batch_size_per_step * Config.group_size
135 |     if accelerator.is_main_process:
136 |         accelerator.print(f"--- My GRPO Auto

## Cell 4：收集阶段——GRPO 的核心前半部分

这是 GRPO 算法的**第 1-4 步**，对应阶段 1 的：

1. **组采样** → `model.generate()` 生成 G 个回答
2. **奖励计算** → `toy_env.compute_reward()` 打分
3. **KL 计算** → `log_prob - ref_log_prob`（K1 估计器）
4. **优势计算** → `(reward - mean) / std`

### 关键代码逐行解读

#### 生成 G 个回答（组采样）
```python
input_ids_repeated = inputs.input_ids.repeat(group_size, 1)  # 把 prompt 复制 G 份
generated_ids = model.generate(input_ids_repeated, ...)       # 一次生成 G 个回答
```

#### 计算 log 概率（为什么用 log？）
```python
log_probs_all = F.log_softmax(logits, dim=-1)                 # softmax 后取 log
token_log_probs = torch.gather(log_probs_all, -1, targets)    # 取出实际生成 token 的 log 概率
```

**为什么用 log 概率而不是直接用概率？**
- 概率值很小（比如 0.0001），多个相乘会下溢
- log 概率是负数，相加代替相乘，数值更稳定
- 这就是概率论里 `log(P(A∩B)) = log P(A) + log P(B)` 的应用

#### KL 散度（K1 估计器）
```python
per_token_kl = token_log_probs - ref_token_log_probs  # 逐 token 的 KL
kl_penalty = (per_token_kl * loss_mask).sum(dim=1)    # 只对生成部分求和
rewards_with_kl = rewards - beta * kl_penalty         # 从奖励中减去 KL 惩罚
```

**K1 估计器**：`KL ≈ log π_θ - log π_ref`。这是无偏估计，比传统 K3（放在 loss 里）更稳定。

#### 优势计算
```python
mean_r = rewards_with_kl.mean()
std_r = rewards_with_kl.std()
advantages = (rewards_with_kl - mean_r) / (std_r + 1e-4)
```

和阶段 1 的 `compute_advantages()` **完全一样**！就是标准化：(x - 均值) / 标准差。

`1e-4` 是防止除以 0（当组内全对或全错时 std=0）。

In [4]:
# 显示收集阶段（第 222-306 行）
print('=== 收集阶段（第 222-306 行）===')
for i, line in enumerate(lines[221:306], start=222):
    print(f'{i:3d} | {line}')

=== 收集阶段（第 222-306 行）===
222 |         # --------------------------------------------------------------------------------
223 |         # 2.2 Collection Phase (Generate and Collect Data)
224 |         # --------------------------------------------------------------------------------
225 |         for i, prompt in enumerate(prompts):
226 |             
227 |             # prepare input for generation (add system prompt, tokenize, and move to device)
228 |             messages = [{"role": "system", "content": Config.system_prompt}, {"role": "user", "content": prompt}]
229 |             try:
230 |                 text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
231 |             except:
232 |                 text_input = f"{Config.system_prompt}\nUser: {prompt}\nAssistant:"
233 |             inputs = tokenizer(text_input, return_tensors="pt").to(device)
234 |             prompt_len = inputs.input_ids.shape[1]
235 |             
236 |        

## Cell 5：训练阶段——PPO 裁剪与梯度更新

这是 GRPO 算法的**第 5-6 步**，对应阶段 1 的策略梯度更新。

### 关键代码逐行解读

#### 概率比（Importance Sampling Ratio）
```python
log_ratio = new_log_probs - train_old_log_probs  # log(π_new) - log(π_old)
ratio = torch.exp(log_ratio)                     # π_new / π_old
```

**为什么取 log 相减再 exp？**
- 直接算 `π_new / π_old` 会因为概率值太小而数值不稳定
- `exp(log(a) - log(b)) = a / b`，用 log 空间计算更稳定
- 这就是高数里 `ln(a/b) = ln(a) - ln(b)` 的应用

#### PPO 裁剪（Clipping）
```python
surr1 = ratio * adv_per_token
surr2 = torch.clamp(ratio, 1-epsilon, 1+epsilon) * adv_per_token
pg_loss = -torch.min(surr1, surr2)
```

和阶段 1 的 `policy_gradient_update()` **完全一样**：
- `surr1`：不裁剪的目标函数
- `surr2`：裁剪后的目标函数（ratio 限制在 [1-ε, 1+ε]）
- `torch.min`：取更悲观的那个（保守更新）
- `torch.clamp` 就是阶段 1 的 `np.clip`

#### 为什么取 min？

取 min 是 PPO 的核心思想——**宁可少更新，不可更新太多**：
- 当 advantage > 0（好回答）：min 限制 ratio 不超过 1+ε，防止过度强化
- 当 advantage < 0（差回答）：min 限制 ratio 不低于 1-ε，防止过度惩罚

#### Loss 聚合
```python
loss = (pg_loss * train_loss_mask).sum() / (train_loss_mask.sum() + 1e-6)
```

`loss_mask` 排除了 prompt 部分和 padding 部分——只对模型生成的 token 计算 loss。

#### inner_update_epochs

```python
for epoch in range(Config.inner_update_epochs):  # 同批数据更新 3 次
    # forward → ratio → clip → loss → backward → step
```

每次更新后，`new_log_probs` 会变化（因为模型参数变了），但 `train_old_log_probs` 不变（收集阶段固定的）。所以 ratio 会逐渐偏离 1.0，裁剪的作用就体现出来了。

In [5]:
# 显示训练阶段（第 329-393 行）
print('=== 训练阶段（第 329-393 行）===')
for i, line in enumerate(lines[328:393], start=329):
    print(f'{i:3d} | {line}')

=== 训练阶段（第 329-393 行）===
329 |         # --------------------------------------------------------------------------------
330 |         # 2.4 Training Phase - with Ratio and Clipping (GRPO core logic)
331 |         # --------------------------------------------------------------------------------
332 |         batch_loss_history, batch_kl_history = [], []
333 |         for epoch in range(Config.inner_update_epochs):
334 |             # Forward pass (Batch)
335 |             outputs = model(input_ids=train_input_ids, attention_mask=train_mask)   # [B*G, Seq, Vocab], becuase the train_input_ids.shape == (batch_size * group_size, max_seq_len)
336 |             logits = outputs.logits[:, :-1, :]                                      # [B*G, Seq-1, Vocab]
337 |             targets = train_input_ids[:, 1:]                                        # [B*G, Seq-1]
338 | 
339 |             new_log_probs_all = F.log_softmax(logits, dim=-1)
340 |             new_log_probs = torch.gather(new_log_probs

## Cell 6：完整流程对比——三个阶段的对应关系

现在你已经看了三个阶段的 GRPO 实现，来对比一下：

### 算法步骤对应

| GRPO 步骤 | 阶段 1（numpy 手写） | 阶段 2（TRL 封装） | 阶段 3（源码阅读） |
|-----------|---------------------|-------------------|-------------------|
| 组采样 | `group_sampling()` | GRPOTrainer 内部 | `model.generate()` × G |
| 奖励计算 | `compute_reward()` | `correctness_reward()` | `toy_env.compute_reward()` |
| KL 计算 | `apply_kl_penalty()` | `beta` 参数控制 | `log_prob - ref_log_prob` |
| 优势计算 | `compute_advantages()` | GRPOTrainer 内部 | `(reward - mean) / std` |
| PPO 裁剪 | `policy_gradient_update()` | GRPOTrainer 内部 | `torch.clamp(ratio, 1-ε, 1+ε)` |
| 梯度更新 | `param += lr * grad` | GRPOTrainer 内部 | `optimizer.step()` |

### 你应该看到的结论

**GRPO 的核心逻辑其实很简单**——三个阶段做的事情完全一样：

1. 生成 G 个回答
2. 打分
3. 标准化（算优势）
4. 裁剪更新

TRL 封装了这些步骤让你几行代码就能跑，但底层逻辑和 400 行的 transparent-grpo 完全一致，也和你阶段 1 手写的 numpy 代码本质相同。

### 阶段 3 相比阶段 2 的新东西

| 新概念 | 说明 |
|--------|------|
| `inner_update_epochs` | 同批数据多次更新，提高数据利用率 |
| K1 KL 估计器 | KL 放在 reward 里而不是 loss 里，更稳定 |
| `log_softmax` + `gather` | 高效提取特定 token 的 log 概率 |
| `accelerate` | HuggingFace 的分布式训练库 |
| `gradient_checkpointing` | 用时间换显存的技术 |
| partial reward | 部分得分增加组内多样性 |

In [6]:
# ============================================================
# 打印完整源码的关键行号索引
# ============================================================

print('=== transparent_grpo.py 结构索引 ===')
print()
sections = [
    (20, 40, 'Config 类', '配置参数'),
    (45, 99, 'ToyEnv 类', '环境和奖励函数'),
    (101, 117, 'MathDataset 类', '数据集'),
    (122, 185, '初始化', '模型加载、优化器、Accelerator'),
    (195, 220, '训练循环开始', '数据准备'),
    (222, 306, '收集阶段', '生成→奖励→KL→优势'),
    (308, 327, 'Buffer 堆叠', 'pad_sequence 拼接'),
    (329, 393, '训练阶段', 'PPO裁剪→loss→backward→step'),
]

for start, end, name, desc in sections:
    print(f'  第 {start:3d}-{end:3d} 行: {name} ({desc})')

print()
print('=== 关键代码行速查 ===')
key_lines = [
    (23, 'group_size = 4', '组大小 G'),
    (33, 'clip_epsilon = 0.2', 'PPO 裁剪范围'),
    (34, 'inner_update_epochs = 3', '同批数据更新次数'),
    (239, 'inputs.input_ids.repeat(group_size, 1)', '组采样：复制 G 份'),
    (243, 'model.generate(...)', '生成 G 个回答'),
    (257, 'toy_env.compute_reward(...)', '计算奖励'),
    (267, 'torch.gather(log_probs_all, ...)', '提取 log 概率'),
    (278, 'per_token_kl = token_log_probs - ref_token_log_probs', 'KL 散度（K1）'),
    (280, 'rewards_with_kl = rewards - beta * kl_penalty', 'KL 惩罚注入奖励'),
    (285, 'advantages = (rewards_with_kl - mean_r) / (std_r + 1e-4)', '优势标准化'),
    (343, 'log_ratio = new_log_probs - train_old_log_probs', 'log 概率比'),
    (344, 'ratio = torch.exp(log_ratio)', '概率比 π_new/π_old'),
    (351, 'torch.clamp(ratio, 1-ε, 1+ε)', 'PPO 裁剪'),
    (352, 'pg_loss = -torch.min(surr1, surr2)', '取 min（保守更新）'),
    (355, 'loss = (pg_loss * mask).sum() / mask.sum()', 'masked loss'),
    (366, 'accelerator.backward(loss)', '反向传播'),
    (367, 'optimizer.step()', '参数更新'),
]

for line_no, code, desc in key_lines:
    print(f'  第 {line_no:3d} 行: {desc}')
    print(f'         {code}')
    print()

=== transparent_grpo.py 结构索引 ===

  第  20- 40 行: Config 类 (配置参数)
  第  45- 99 行: ToyEnv 类 (环境和奖励函数)
  第 101-117 行: MathDataset 类 (数据集)
  第 122-185 行: 初始化 (模型加载、优化器、Accelerator)
  第 195-220 行: 训练循环开始 (数据准备)
  第 222-306 行: 收集阶段 (生成→奖励→KL→优势)
  第 308-327 行: Buffer 堆叠 (pad_sequence 拼接)
  第 329-393 行: 训练阶段 (PPO裁剪→loss→backward→step)

=== 关键代码行速查 ===
  第  23 行: 组大小 G
         group_size = 4

  第  33 行: PPO 裁剪范围
         clip_epsilon = 0.2

  第  34 行: 同批数据更新次数
         inner_update_epochs = 3

  第 239 行: 组采样：复制 G 份
         inputs.input_ids.repeat(group_size, 1)

  第 243 行: 生成 G 个回答
         model.generate(...)

  第 257 行: 计算奖励
         toy_env.compute_reward(...)

  第 267 行: 提取 log 概率
         torch.gather(log_probs_all, ...)

  第 278 行: KL 散度（K1）
         per_token_kl = token_log_probs - ref_token_log_probs

  第 280 行: KL 惩罚注入奖励
         rewards_with_kl = rewards - beta * kl_penalty

  第 285 行: 优势标准化
         advantages = (rewards_with_kl - mean_r) / (std_r + 1e-4)

  第 343 行: log 概率比
      

## Cell 7：理解检查点

回答以下问题确认你理解了源码：

1. **能在代码中指出哪一行计算优势吗？**
   - 提示：第 285 行，`(rewards_with_kl - mean_r) / (std_r + 1e-4)`

2. **能在代码中指出哪一行计算 KL 散度吗？**
   - 提示：第 278 行，`per_token_kl = token_log_probs - ref_token_log_probs`

3. **能解释为什么取 log 概率而不是直接用概率吗？**
   - 提示：数值稳定性，log 空间相加代替相乘

4. **`inner_update_epochs=3` 意味着什么？为什么需要 PPO 裁剪？**
   - 提示：同批数据更新 3 次，每次更新后策略偏离旧策略，裁剪防止偏离过大

5. **K1 估计器和阶段 1 的 KL 计算有什么区别？**
   - 提示：K1 把 KL 放在 reward 里（`rewards - beta * kl`），阶段 1 放在 loss 里

6. **`loss_mask` 的作用是什么？为什么要排除 prompt 部分？**
   - 提示：只对模型生成的 token 计算 loss，prompt 是固定的不应该影响更新

7. **`torch.gather` 在代码中做了什么？**
   - 提示：从 log_softmax 的输出中，取出实际生成 token 对应的 log 概率

### 动手实验建议

```python
# 实验 1：修改 KL 估计器
# 把 K1（reward 里减 KL）改成 K3（loss 里加 KL），对比训练稳定性

# 实验 2：修改 inner_update_epochs
# 从 3 改成 1 或 10，观察 reward 曲线变化

# 实验 3：修改 clip_epsilon
# 从 0.2 改成 0.1 或 0.3，观察训练稳定性
```

### 下一步

如果你能回答以上所有问题，恭喜！你已经理解了 GRPO 的代码实现。

接下来在**阶段 4**中，我们会在 4090 服务器上用更大的模型（1.5B/3B）和 vLLM 加速做正式训练。